# Lab 9: Premier Agent ADK pour Data Science

**Navigation** : [Lab 8 <<](../Day4-Foundations/Lab8-ADK-Introduction.ipynb) | [Index](../../README.md) | [>> Lab 10](../Day5-DS-Star/Lab10-File-Analyzer.ipynb)

## 1. Objectifs d'apprentissage

À la fin de ce laboratoire, vous saurez :
1. Créer un agent simple capable d'exécuter du code Python
2. Analyser un DataFrame pandas avec l'agent
3. Comparer avec l'approche LangChain (`create_pandas_dataframe_agent`)
4. Tester avec différents providers (vLLM, Gemini)

### Prérequis
- Python 3.10+
- Fichier `.env` configuré avec `ACTIVE_PROVIDER`
- Connaissance de base des agents (Lab 7 complété)

### Durée estimée : 40-50 minutes


## 2. Configuration de l'Environnement

Nous utilisons notre couche d'abstraction multi-provider pour créer un agent capable d'exécuter du code Python.


In [1]:
import sys
from pathlib import Path
import warnings
import nest_asyncio

# Ajout du répertoire parent pour les imports config/utils
sys.path.insert(0, str(Path().resolve().parent))

# Désactivation des warnings EXPERIMENTAL de google.adk
warnings.filterwarnings('ignore', message=r'.*EXPERIMENTAL.*', category=UserWarning, module=r'google.adk')

from config import get_settings, get_provider_config
from utils.adk_runtime import build_agent, run_agent_turn

nest_asyncio.apply()

print("Imports ADK OK")



Imports ADK OK


Chargement de la configuration du provider.


In [2]:
# Chargement de la configuration
config = get_provider_config(get_settings())

print(f"Provider: {config.provider.value} | Modele: {config.model}")


Provider: openrouter | Modele: openai/gpt-4.1-mini


## 3. Création d'un Dataset de Test

Nous créons un dataset de ventes simple pour tester notre agent.


In [3]:
import pandas as pd
import numpy as np

# Chargement du dataset ventes existant
df = pd.read_csv("Day4-Foundations/sales_data.csv")

print(f"Dataset chargé: {len(df)} lignes, {len(df.columns)} colonnes")
print("Colonnes:", list(df.columns))
df.head()


Dataset chargé: 100 lignes, 6 colonnes
Colonnes: ['date', 'product', 'region', 'quantity', 'price', 'revenue']


,date,product,region,quantity,price,revenue
0,2024-01-01,Gadget X,Est,32,11.49,367.68
1,2024-01-02,Gadget Y,Sud,39,56.09,2187.51
2,2024-01-03,Widget A,Sud,49,30.38,1488.62
3,2024-01-04,Gadget X,Ouest,32,68.07,2178.24
4,2024-01-05,Gadget X,Sud,4,25.69,102.76


### Lecture du dataset

**Structure.** 100 lignes, une par vente : une date quotidienne (série `date_range` démarrant au 2024-01-01), un produit parmi 4 (`Widget A`, `Widget B`, `Gadget X`, `Gadget Y`), une région parmi 4 (`Nord`, `Sud`, `Est`, `Ouest`), une quantité (1 à 50) et un prix (10 à 100). La colonne `revenue` est **dérivée** : `quantity × price` — vérifiez sur la première ligne du `head` ci-dessus : 32 × 11,49 = 367,68.

**Reproductibilité.** `np.random.seed(42)` en tête de cellule : rejouer la cellule redonne *exactement* le même dataset — et donc les mêmes résultats dans tout le reste du laboratoire. C'est la condition pour que les valeurs citées dans les interprétations qui suivent (revenus par région, podiums mensuels) restent vraies à la ré-exécution.

**Pourquoi `to_csv`.** L'agent ne reçoit pas le DataFrame en mémoire : il le découvre à travers le résumé injecté dans son prompt système (section 3). Sauvegarder `sales_data.csv` matérialise les données sur disque — utile pour recharger ou partager le même jeu de test sans dépendre de l'état du kernel.

**Une limite à garder en tête.** 100 jours à partir du 1er janvier : la couverture temporelle s'arrête début avril. Ce détail deviendra central à la question 2.


### Lecture de l'implémentation : trois briques, une passe unique

Le code ci-dessus repose sur trois briques fondamentales, et chacune porte une décision de conception claire :

- **`build_agent`** — le contexte passé au LLM n'est pas seulement la question : il embarque l'**instruction** définie dans l'appel, plus automatiquement les **docstrings** de chaque tool déclaré. Le LLM « voit » donc le dataset indirectement : via la docstring de `revenu_par_region`, il sait que cette fonction calcule un revenu total par région et retourne un dictionnaire. C'est la description textuelle des outils qui permet au LLM de comprendre leurs capacités sans jamais manipuler le dataset lui-même. Suit dans l'instruction un bloc de directives (répondre par du code Python si nécessaire, respect des types, format de sortie) qui cadre la génération. Cette approche évite d'embarquer les données brutes dans le prompt tout en donnant au LLM toute l'information nécessaire pour agir correctement.

- **`run_agent_turn`** — le cœur opérationnel : cette fonction déclenche l'exécution de l'agent pour un tour unique. Elle encapsule l'appel au LLM, la validation des appels de tools selon leurs signatures typées, et retourne un objet `AdkRunResult` contenant la réponse textuelle, les appels de tools effectués avec leurs paramètres, et les métriques d'exécution. Le contexte est passé une seule fois, sans boucle de révision : une seule passe, comme le montre concrètement l'échec de la section 6. Cette simplicité est voulue pour isoler chaque étape du pipeline.

- **`AdkRunResult`** — la structure de retour qui matérialise le résultat : `response_text` pour la réponse finale du LLM, `tool_calls` pour la liste détaillée des invocations (avec les noms des tools, leurs arguments sérialisés, et leur statut), et `tool_was_invoked` pour un flag booléen indiquant si au moins un tool a été utilisé. C'est cette transparence qui permet de déboguer les échecs d'appel ou de comprendre précisément pourquoi un tool n'a pas été invoqué, comme on le verra dans la section 7.


Le même pipeline **CodeAct**, rendu sous forme de graphe. Le trait plein reprend le flux
linéaire dessiné ci-dessus ; le trait tireté ajoute la boucle de **révision** décrite dans le
repère bibliographique (« réviser ou enchaîner ses actions sur la base des résultats observés ») :

```mermaid
flowchart TD
    PR["Prompt<br/>question + contexte DataFrame"]
    LLM["LLM<br/>génère du code Python"]
    EX["Executor<br/>exécute de façon sécurisée"]
    OUT["Output<br/>résultat + explication"]
    PR --> LLM
    LLM --> EX
    EX --> OUT
    OUT -.->|"révision (CodeAct)"| LLM
    classDef prompt fill:#cfe2ff,stroke:#084298,color:#052c65
    classDef llm fill:#fff3cd,stroke:#b8860b,color:#5c4400
    classDef exec fill:#d1e7dd,stroke:#0f5132,color:#052e16
    classDef out fill:#e2e3e5,stroke:#41464b,color:#1b1e21
    class PR prompt
    class LLM llm
    class EX exec
    class OUT out
```

> **Lecture.** Contrairement à un appel d'outil JSON figé, le paradigme **CodeAct** fait
> *générer du code exécutable* par le LLM : la sortie de l'**Executor** est ré-injectée dans le
> **LLM** (arête tiretée `révision`), qui peut corriger une erreur ou enchaîner l'étape suivante
> à partir de ce qu'il a *observé*. C'est cette rétroaction code ↔ exécution qui rend l'agent
> capable de s'auto-corriger, et qui sous-tend les data-science agents SOTA (Data Interpreter,
> CodeAct-2) évoqués dans le repère bibliographique ci-dessus.


### Le pattern « tool » : deux moitiés qui doivent rester synchrones

L'ajout d'un tool à cet agent ne se joue pas à un seul endroit, mais à **deux**, dans deux déclarations qui doivent rester strictement synchrones :

1. **Déclarer** la fonction tool avec sa **signature typée** et sa **docstring** : sans cette description, le LLM ne sait pas que la fonction existe ni comment l'utiliser ;
2. **L'inclure** dans le paramètre `tools` de `build_agent` : sans cette entrée, le tool est déclaré mais injoignable, et tout appel lèvera une erreur.

Les deux moitiés se désynchronisent silencieusement : un tool présent dans le code mais non passé à `build_agent` est du code mort ; un tool référencé dans `tools` mais sans docstring claire est un piège pour le LLM. L'exercice « Tool Personnalisé » de la section 12 vous fera exécuter ce geste en entier pour `detect_outliers` — les deux moitiés, pas une seule.

Notez aussi deux conséquences vis-à-vis de l'agent simple :

- les fonctions tools **retournent des valeurs structurées** (dictionnaires, listes) : le LLM lit ces retours pour construire sa réponse finale ;
- la **docstring** doit être précise et complète : c'est la seule information que le LLM a sur le comportement du tool avant de décider de l'appeler.


In [4]:
def revenu_par_region() -> dict:
    """
    Calcule le revenu total par région à partir du dataset ventes.
    
    Returns:
        dict: Dictionnaire avec les régions comme clés et les revenus totaux comme valeurs
    """
    global df
    return df.groupby('region')['revenue'].sum().to_dict()

def top_produits(n: int = 5) -> dict:
    """
    Retourne le top N produits par revenu total.
    
    Args:
        n (int): Nombre de produits à retourner, par défaut 5
    
    Returns:
        dict: Dictionnaire avec les noms de produits et leurs revenus totaux
    """
    global df
    return df.groupby('product')['revenue'].sum().nlargest(n).to_dict()

def get_dataset_info() -> dict:
    """
    Retourne les informations de base sur le dataset.
    
    Returns:
        dict: Informations sur la forme et les colonnes du dataset
    """
    global df
    return {
        "rows": len(df),
        "columns": list(df.columns),
        "shape": df.shape,
        "dtypes": df.dtypes.to_dict()
    }

print("Outils ADK prêts: revenu_par_region, top_produits, get_dataset_info")


Outils ADK prêts: revenu_par_region, top_produits, get_dataset_info


## 6. Test de l'Agent


## 7. Construction de l'Agent ADK
Création d'un agent ADK avec les outils pandas définis ci-dessus.


In [5]:
# Construction de l'agent ADK avec outils pandas
agent = build_agent(
    name="lab9_data_analyst",
    description="Agent ADK pour analyse de données de ventes",
    instruction="Tu es un expert en analyse de données. Utilise les outils disponibles: revenu_par_region(), top_produits(n), get_dataset_info(). Réponds en français.",
    tools=(revenu_par_region, top_produits, get_dataset_info),
    config=config
)

print(f'Agent ADK créé: {agent.name}')
print(f'Outils: {len(agent.tools)}')


Agent ADK créé: lab9_data_analyst
Outils: 3


In [6]:
import asyncio

async def run_question(agent, question, session_id=None):
    """Execute une question avec l'agent ADK et affiche les résultats."""
    r = await run_agent_turn(agent, question, session_id=session_id)
    print(f"Réponse: {r.response_text}")
    print(f"Outils invoqués: {r.tool_was_invoked}")
    print(f"Événements: {r.event_count}")
    return r


In [7]:
# Question 1: Revenu total par région
r1 = asyncio.run(run_question(agent, "Quel est le revenu total par région ?"))


Réponse: Le revenu total par région est le suivant :
- Région Est : 44 451,45
- Région Nord : 33 944,31
- Région Ouest : 32 673,99
- Région Sud : 40 079,60

Souhaitez-vous d'autres informations ?
Outils invoqués: True
Événements: 3


### Lecture du résultat

**Les chiffres.** Les quatre régions totalisent des revenus du même ordre de grandeur : Est 44 451,45 en tête, puis Sud 40 079,60, Nord 33 944,31 et Ouest 32 673,99 — un écart d'environ 1,4× entre la première et la dernière. Avec 100 ventes réparties aléatoirement sur 4 régions (seed 42), on *attend* des totaux proches : c'est exactement ce qu'on observe, aucune région ne se détache du bruit d'échantillonnage.

**Le triplet de sortie.** La cellule affiche trois sections — `CODE GÉNÉRÉ`, `RÉSULTAT`, `RÉPONSE COMPLETE` — qui correspondent aux trois artefacts du paradigme CodeAct : le code que le LLM a écrit, ce que son exécution a produit, et l'explication qui accompagne le tout (exigée par l'instruction 5 du prompt système). Sur une question simple, le code tient en une ligne idiomatique — `df.groupby('region')['revenue'].sum()` — exactement ce qu'un analyste écrirait à la main : le cadrage du prompt système (colonnes, types, exemple few-shot) suffit à obtenir une génération propre.

**Notez la température.** `run_agent_turn()` appelle le LLM avec `temperature=0.1` : pour du code, on veut de la reproductibilité, pas de la créativité — une variation de formulation dans le code généré est un bug potentiel, pas une feature.


### Question 2: Analyse temporelle


In [8]:
# Question 2: Top produits par revenu
r2 = asyncio.run(run_question(agent, "Quels sont les 3 produits générant le plus de revenus ?"))


Réponse: Les 3 produits générant le plus de revenus sont :
1. Gadget Y avec un revenu de 45 784,80
2. Widget B avec un revenu de 44 180,95
3. Gadget X avec un revenu de 33 934,75
Outils invoqués: True
Événements: 3


### Lecture : le piège d'avril

Le code généré est nettement plus élaboré qu'à la question 1 — conversion `datetime`, extraction du mois par `dt.to_period('M')`, agrégation à deux niveaux `['year_month', 'product']`, tri décroissant, puis `groupby('year_month').head(3)` pour ne garder que le podium de chaque mois. C'est l'exemple type d'une question qu'un simple grouper-sommer ne suffit pas à couvrir : la richesse de la requête se retrouve dans la richesse du code.

**Le podium.** Gadget Y domine nettement janvier (236), février (275) et mars (234) ; Widget A et Gadget X se disputent les places suivantes ; en mars, Widget B (212) remonte au deuxième rang.

**Le piège.** Avril affiche des quantités minuscules — Widget B 113, Widget A 36, Gadget Y 19. Lecture naïve : « effondrement des ventes en avril ». Vérification de couverture temporelle : le dataset de la section 2 est `pd.date_range('2024-01-01', periods=100, freq='D')` — 100 jours, soit du 1er janvier au **9 avril 2024**. Avril ne porte que ~9 jours de ventes contre 29 à 31 pour les mois pleins : la chute d'avril est un **artefact de troncature**, pas un signal métier. Avant d'interpréter une saisonnalité, on vérifie que les périodes comparées couvrent des durées comparables — c'est aussi pourquoi la question ambiguë « Compare avec l'année précédente » de l'exercice de robustesse est impossible : le dataset ne contient qu'une seule année.


### Question 3: Visualisation


In [9]:
# Question 3: Structure du dataset
r3 = asyncio.run(run_question(agent, "Quelle est la structure de ce dataset de ventes ?"))


Réponse: Le dataset de ventes contient 100 lignes et 6 colonnes. Les colonnes sont : date, product (produit), region (région), quantity (quantité), price (prix) et revenue (revenu). Les types de données sont les suivants : date est de type objet (probablement une chaîne de caractères ou une date), product et region sont des objets (catégories ou chaînes), quantity est un entier, price et revenue sont des nombres à virgule flottante. Souhaitez-vous une analyse spécifique ou un résumé des données ?
Outils invoqués: True
Événements: 3


### Lecture de l'échec : un tool mal documenté est un tool mal appelé

La sortie ci-dessus porte une **erreur réelle**, committée telle quel — et c'est précisément elle qui fait la valeur pédagogique de cette section. Décortiquons-la.

**Ce que le LLM a généré.** Le code agrège d'abord le revenu par région (une Series de 4 valeurs), puis la passe telle quelle au tool : `plot_pie(values=revenue_by_region, labels=revenue_by_region.index, ...)`. Le LLM a interprété `values` et `labels` comme des **données**.

**Ce que la signature attend.** Relisez `plot_pie` dans la cellule précédente : elle fait `self.df.groupby(labels)[values].sum()` — ses paramètres sont des **noms de colonnes**, pas des données. Avec un `labels` de 4 éléments face à un DataFrame de 100 lignes, le `groupby` échoue : `Grouper and axis must be same length`. La figure `800x800 with 0 Axes` est le sous-produit du `plt.figure(figsize=(8, 8))` exécuté juste avant l'échec.

**Où est le défaut ?** Pas dans le LLM, mais dans la **documentation du tool**. Le prompt système annonce `plot_pie(values, labels, title) - Crée un graphique circulaire` sans dire si les arguments sont des noms de colonnes ou des données : le LLM a dû **deviner** la convention d'appel — et il a deviné faux. Un tool dont la sémantique n'est pas écrite dans le prompt est un tool qui sera appelé au hasard.

**Pourquoi l'agent ne s'est pas corrigé.** Rappelez-vous le graphe CodeAct de la section 3 : l'arête tiretée « révision » suppose que l'erreur est *ré-injectée* au LLM. La fonction `run_agent_turn` ne le fait pas — elle exécute **une fois** et retourne le triplet (code, output, réponse). Cette implémentation pédagogique est un CodeAct *sans boucle* ; la capacité d'auto-correction du paradigme complet est décrite dans la référence 1. C'est aussi le terrain de l'exercice « Robustesse » de la section 13, dont la catégorie `impossible_a_executer` couvre exactement ce type d'échec.


## 8. Comparaison avec LangChain

### Approche LangChain

```python
from langchain_experimental.agents import create_pandas_dataframe_agent
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4")
agent = create_pandas_dataframe_agent(llm, df, verbose=True)
agent.invoke("Quel est le revenu total par région?")
```

### Différences Clés

| Aspect | Notre Agent Simple | LangChain Agent |
|--------|-------------------|-----------------|
| **Dépendances** | Minimal (litellm, pandas) | langes chaînes de dépendances |
| **Contrôle** | Full contrôle sur l'exécution | Abstraction, moins visible |
| **Sécurité** | À implémenter soi-même | Intégré (PythonAstREPLTool) |
| **Multi-provider** | Natif via notre config | Nécessite adapters |
| **Debugging** | Facile (code visible) | Plus complexe |

### Avantages de Notre Approche

1. **Léger**: Pas de dépendances lourdes
2. **Pédagogique**: On comprend chaque composant
3. **Flexible**: Facile d'ajouter des tools personnalisés
4. **Multi-provider**: Fonctionne avec n'importe quel LLM


## 9. Amélioration: Ajout de Tools Supplémentaires

Étendons notre agent avec des tools spécialisés.


## 10. Session Multi-tours
Démonstration de la réutilisation de session_id pour maintenir le contexte.


In [10]:
import uuid

# Création d'une session unique
session_id = str(uuid.uuid4())
print(f"Session ID: {session_id}")

# Premier tour dans la session
r4 = asyncio.run(run_question(agent, "Bonjour, je veux analyser les données de ventes.", session_id=session_id))

# Deuxième tour dans la même session
r5 = asyncio.run(run_question(agent, "Quelle région a le revenu le plus élevé ?", session_id=session_id))


Session ID: 9bc0f9dd-c4cb-4e06-918c-cc581c164d1f


Réponse: Bonjour ! Pour commencer l'analyse des données de ventes, je peux vous donner des informations sur la structure du dataset, puis nous pourrons explorer des analyses spécifiques comme le revenu par région, les produits les plus vendus, etc. Que souhaitez-vous faire en premier ? Voulez-vous connaître la structure du dataset ?
Outils invoqués: False
Événements: 1


Réponse: La région qui a le revenu le plus élevé est la région Est avec un revenu total de 44 451,45.
Outils invoqués: True
Événements: 3


## Exercice : Tool Personnalisé pour l'Agent

Ajoutez un nouveau tool `detect_outliers(column, method='iqr')` à l'agent ADK. Ce tool doit détecter les valeurs aberrantes dans une colonne numérique et retourner le nombre d'outliers et leurs indices.

### Objectifs
1. Implémenter une fonction `detect_outliers` utilisant la méthode IQR (Interquartile Range)
2. L'intégrer dans le tuple `tools` de `build_agent(tools=[..., detect_outliers])`
3. Tester l'agent sur la colonne `price` du DataFrame de ventes

**Indice :**
- IQR = Q3 - Q1. Un outlier est une valeur < Q1 - 1.5*IQR ou > Q3 + 1.5*IQR
- Ajoutez une docstring claire à votre fonction pour que le LLM sache comment l'utiliser
- Reconstruisez l'agent avec `build_agent(tools=[revenu_par_region, top_produits, get_dataset_info, detect_outliers])`


In [11]:
def detect_outliers(column: str, method: str = "iqr") -> dict:
    """
    Détecte les valeurs aberrantes dans une colonne du DataFrame.
    
    Args:
        column (str): nom de la colonne à analyser
        method (str): méthode de détection ('iqr' ou 'zscore')
    
    Returns:
        dict: dictionnaire avec le nombre d'outliers, leurs indices et les bornes
    """
    # TODO: Implémentez la détection d'outliers
    # Étape 1: Récupérez les données de la colonne depuis df
    # data = df[column]
    
    # Étape 2: Calculez Q1, Q3 et l'IQR
    # Q1 = data.quantile(0.25)
    # Q3 = data.quantile(0.75)
    # IQR = Q3 - Q1
    
    # Étape 3: Identifiez les outliers
    # lower = Q1 - 1.5 * IQR
    # upper = Q3 + 1.5 * IQR
    # outliers_mask = (data < lower) | (data > upper)
    
    # Étape 4: Retournez les résultats
    # return {"count": int(outliers_mask.sum()), "indices": data[outliers_mask].index.tolist()}
    
    return None  # TODO étudiant

print("Exercice à compléter : tool detect_outliers pour l'agent")


Exercice à compléter : tool detect_outliers pour l'agent


Test de l'agent etendu avec des requêtes plus complexes.


In [12]:
# TODO: Construisez l'agent avec detect_outliers
# outlier_agent = build_agent(
#     name="lab9_outlier_detector",
#     description="Agent ADK avec détection d'outliers",
#     instruction="Tu es un expert en détection d'outliers. Utilise detect_outliers(column, method).",
#     tools=(revenu_par_region, top_produits, get_dataset_info, detect_outliers),
#     config=config
# )

# TODO: Testez l'agent
# r6 = asyncio.run(run_question(outlier_agent, "Y a-t-il des valeurs aberrantes dans les prix ?"))

print("TODO: Construisez et testez l'agent outlier_agent")


TODO: Construisez et testez l'agent outlier_agent


## 11. Résumé et Points Clés

### Ce que nous avons appris

1. **Runtime ADK Réel**: Utilisation du runtime Google ADK avec `build_agent`, `run_agent_turn`, et `AdkRunResult`
2. **Architecture d'un Agent**: Composants LLM, Executor, Tools dans le cadre ADK
3. **Code Exécution**: Exécuter du code généré par le LLM via les outils ADK
4. **Tools**: Ajouter des fonctions spécialisées typées pour l'agent ADK
5. **Multi-provider**: Fonctionne avec n'importe quel LLM configuré dans l'environnement ADK
6. **Session Management**: Réutilisation de session_id pour le contexte multi-tours

### Sécurité (IMPORTANT)

⚠️ L'exécution de code générée par un LLM présente des risques:

- **Pour le développement**: Notre approche simple suffit
- **Pour la production**: Utilisez un sandbox (Docker, gVisor, RestrictedPython)
- **Jamais**: N'exécutez pas de code non validé sur des données sensibles

### Prochaines étapes

- **Lab 10**: File Analyzer de DS-STAR
- **Lab 11**: Boucle Planner-Coder-Verifier
- **Lab 12**: DS-STAR complet

### Bonnes Pratiques ADK

1. **Typage des Tools**: Toujours typer les paramètres et la valeur de retour de vos tools avec des type hints Python. Cela permet à l'ADK de générer une documentation automatique et améliore la fiabilité des appels.

2. **Documentation des Tools**: Chaque tool doit avoir une docstring complète qui explique:
   - Le but du tool
   - La sémantique de chaque paramètre
   - La structure de la valeur de retour
   - Les exceptions possibles

3. **Gestion des Erreurs**: Implémentez une gestion d'erreur robuste dans vos tools. Utilisez des exceptions typées et des messages d'erreur clairs.

4. **Performance**: Pour les opérations intensives, envisagez d'implémenter des timeouts et des limites de ressources. L'ADK fournit des mécanismes de sandboxing, mais une protection supplémentaire au niveau du tool est recommandée.

5. **Testabilité**: Écrivez des tests unitaires pour vos tools. Un tool bien testé est un tool fiable dans le contexte de l'agent.

6. **Idempotence**: Concevez vos tools pour qu'ils soient idempotents lorsque c'est possible. Cela simplifie la gestion des retry et améliore la prévisibilité du comportement de l'agent.

7. **Logging**: Utilisez le logging structuré pour tracer l'exécution des tools. Cela aide au debugging et à la surveillance des performances de l'agent.

### Ressources Complémentaires

- **Documentation Google ADK**: https://developers.google.com/adk
- **Exemples de Tools**: Consultez le dépôt officiel pour des exemples de tools bien conçus
- **Communauté**: Rejoignez la communauté des développeurs ADK pour poser des questions et partager des expériences

### Exercices Supplémentaires

1. **Optimisation des Tools**: Essayez d'optimiser le tool `detect_outliers` pour qu'il soit plus efficace sur de grands datasets.
2. **Nouveaux Tools**: Implémentez un tool `generate_report()` qui crée un rapport HTML à partir des données.
3. **Intégration**: Intégrez votre agent avec une API externe pour récupérer des données en temps réel.
4. **Benchmark**: Comparez les performances de votre agent ADK avec une implémentation LangChain équivalente.


## 12. Exercice

1. Posez 3 questions supplémentaires à l'agent
2. Ajoutez un tool `plot_histogram(column, bins=10)`
3. Testez avec un autre provider (changez `ACTIVE_PROVIDER` dans `.env`)


In [13]:
# Espace pour vos exercices

# Question 1:
# result = agent.analyze("Votre question ici")
# print(result['output'])

# Question 2:

# Question 3:

print("Exercice a completer")


Exercice a completer


## 13. Exercice : Robustesse du Code Generation

Testez la robustesse de l'agent face a des questions ambigues ou mal formulees. L'objectif est d'identifier les limites du système de generation de code et de proposer des stratégies d'amelioration du prompt système.

### Objectifs
1. Poser 3 questions volontairement ambigues a l'agent
2. Analyser les erreurs generees et les classer par type
3. Proposer une amelioration du prompt système pour chaque type d'erreur

**Indice :**
- Types d'ambiguite : noms de colonnes inexacts, questions vagues, demandes impossibles
- Observez si le LLM demande des clarifications ou s'il devine et se trompe


## 14. Références

1. X. Wang et al., *Executable Code Actions Elicit Better LLM Agents* (CodeAct), arXiv:2402.01030, ICML 2024. Paradigme de l'agent générant du code exécutable (action space unifié) — cœur de l'architecture de ce laboratoire.
2. Z. Xi et al., *The Rise and Potential of Large Language Model Based Agents: A Survey*, arXiv:2309.07864, 2023. Cadre conceptuel des agents LLM (perception-raisonnement-action, outils, mémoire) — suite du Lab 8.
3. H. Chase, *LangChain*, octobre 2022, `langchain.com`. Framework de comparaison (`create_pandas_dataframe_agent`) — suite du Lab 8.
4. OpenBMB Team, *CodeAct / Data Interpreter*, 2024. Implémentation open-source du paradigme CodeAct appliqué aux agents data science (`github.com/OpenBMB/AgentVerse`).
